# AutoEval: Generate Test Pipelines from App Descriptions

Map your AI application's capabilities to the right eval metrics, build a reusable test pipeline with built-in and custom evals, and automate it in CI/CD.

| Time | Difficulty | Features Used |
|------|-----------|---------------|
| 30 min | Intermediate | Evaluation, Custom Evals, Dataset, CI/CD Pipeline |

You're building **LexAI**, a legal document assistant that helps lawyers draft contracts, summarize case law, extract key clauses, and check for missing provisions. The team knows they need evals but doesn't know *which* evals to use. There are 72+ built-in metrics — which ones matter for legal document generation?

This cookbook teaches you how to think about eval selection systematically: describe your app, map capabilities to metrics, build a test dataset, create domain-specific custom evals, wire everything into a reusable pipeline, and automate it.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/main/use-cases/auto-eval-pipeline.ipynb)

**Prerequisites**
- FutureAGI account → [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` (see Get your API keys)
- OpenAI API key (`OPENAI_API_KEY`)
- Python 3.9+

In [ ]:
!pip install ai-evaluation futureagi openai

In [ ]:
import os

os.environ["FI_API_KEY"] = "your-fi-api-key"
os.environ["FI_SECRET_KEY"] = "your-fi-secret-key"
os.environ["OPENAI_API_KEY"] = "your-openai-key"

## Step 1: Describe your application

Before picking any eval metrics, write down what your AI application actually does. Each capability has different quality requirements, which means different eval metrics.

LexAI has four core capabilities:

| Capability | What it does | Quality risk |
|---|---|---|
| **Contract drafting** | Generate contract clauses from requirements | Missing provisions, fabricated terms |
| **Case summarization** | Summarize court opinions and legal precedents | Omitting key holdings, misrepresenting rulings |
| **Clause extraction** | Pull specific clauses from long contracts | Wrong clause, partial extraction |
| **Missing provision detection** | Flag what a contract is missing | False negatives (missing a gap), false positives |

This is the starting point for every eval pipeline. You can't pick the right metrics until you know what "wrong" looks like for each capability. A hallucinated clause in a contract is a different failure mode than an incomplete case summary — and they need different metrics to catch them.

## Step 2: Map capabilities to eval metrics

Now map each capability to the built-in metrics that catch its specific failure modes. This is the decision matrix that drives your entire pipeline.

| Capability | Metrics | Why these metrics |
|---|---|---|
| Contract drafting | `completeness`, `factual_accuracy`, `groundedness` | Drafts must include all required provisions (completeness), state legally accurate terms (factual_accuracy), and stay grounded in the source requirements (groundedness) |
| Case summarization | `completeness`, `context_relevance`, `chunk_utilization` | Summaries must capture all key holdings (completeness), stay relevant to the query (context_relevance), and use the full source material, not just the first paragraph (chunk_utilization) |
| Clause extraction | `chunk_attribution`, `factual_accuracy` | Extracted clauses must trace back to specific sections in the source document (chunk_attribution) and accurately reproduce the clause content (factual_accuracy) |
| Missing provision detection | `completeness`, custom eval | Detection must cover all standard provisions (completeness), but "standard provisions" is domain-specific — that needs a custom eval |

Three metrics appear across multiple capabilities: `completeness`, `factual_accuracy`, and `groundedness`. These are your core quality signals for any legal document task. The others (`context_relevance`, `chunk_utilization`, `chunk_attribution`) are RAG-specific metrics that matter when the agent retrieves and synthesizes from source documents.

> **Note:** Not sure which built-in metrics exist? See the full metric reference for all 72+ metrics with required keys and output types. For a hands-on introduction to the `evaluate()` function, see Running Your First Eval.

## Step 3: Build your test dataset

Create a test dataset that covers all four capabilities with realistic legal examples. Each row needs an `input` (the task), `context` (source material the agent should draw from), and `output` (the agent's response to evaluate).

In [ ]:
TEST_CASES = [
    {
        "capability": "contract_drafting",
        "input": "Draft a confidentiality clause for a software licensing agreement between TechVentures Inc and DataFlow Corp.",
        "context": (
            "The agreement covers proprietary source code and trade secrets. "
            "The confidentiality period is 5 years post-termination. "
            "Both parties must restrict access to employees with a need-to-know basis. "
            "Breach of confidentiality allows the non-breaching party to seek injunctive relief and damages."
        ),
        "output": (
            "CONFIDENTIALITY. Each party agrees to hold in strict confidence all Confidential Information "
            "of the other party, including but not limited to proprietary source code and trade secrets. "
            "Access to Confidential Information shall be limited to employees with a demonstrated need-to-know. "
            "This obligation shall survive for a period of five (5) years following termination of this Agreement. "
            "In the event of a breach, the non-breaching party shall be entitled to seek injunctive relief "
            "and monetary damages in addition to any other remedies available at law or in equity."
        ),
    },
    {
        "capability": "contract_drafting",
        "input": "Draft a termination clause for a SaaS subscription agreement.",
        "context": (
            "Either party may terminate with 30 days written notice. "
            "The provider may terminate immediately for non-payment after a 15-day cure period. "
            "Upon termination, the customer's data must be exported within 60 days, after which it will be deleted."
        ),
        "output": (
            "TERMINATION. Either party may terminate this Agreement by providing thirty (30) days prior "
            "written notice to the other party. Notwithstanding the foregoing, Provider may terminate this "
            "Agreement immediately upon written notice if Customer fails to cure any payment default within "
            "fifteen (15) days of receiving notice of such default. Upon termination, Customer shall have "
            "sixty (60) days to export all Customer Data, after which Provider shall permanently delete "
            "all Customer Data from its systems."
        ),
    },
    {
        "capability": "case_summarization",
        "input": "Summarize the key holding and reasoning in this trade secret case.",
        "context": (
            "In Waymo LLC v. Uber Technologies Inc (2018), the court addressed whether a former employee's "
            "download of 14,000 proprietary files constituted trade secret misappropriation under the DTSA. "
            "The court held that downloading files alone, combined with subsequent employment at a competitor, "
            "established a prima facie case of misappropriation. The court emphasized that the Defend Trade "
            "Secrets Act does not require proof of actual use — acquisition through improper means is sufficient. "
            "The case settled for approximately $245 million in equity."
        ),
        "output": (
            "In Waymo v. Uber (2018), the court ruled that downloading proprietary files and subsequently "
            "joining a competitor established a prima facie case of trade secret misappropriation under the DTSA. "
            "The key holding was that the Act does not require proof of actual use of the trade secrets — "
            "acquisition through improper means alone is sufficient. The case settled for approximately "
            "$245 million in Uber equity."
        ),
    },
    {
        "capability": "case_summarization",
        "input": "Summarize this case on non-compete enforceability.",
        "context": (
            "In Edwards v. Arthur Andersen LLP (2008), the California Supreme Court held that non-compete "
            "agreements are void under California Business and Professions Code Section 16600, with narrow "
            "exceptions limited to the sale of a business or dissolution of a partnership. The court rejected "
            "the 'narrow restraint' doctrine that some lower courts had applied, ruling that Section 16600 "
            "must be read broadly to void any agreement that restrains a person from engaging in a lawful "
            "profession, trade, or business. The decision reinforced California's strong public policy "
            "favoring employee mobility."
        ),
        "output": (
            "In Edwards v. Arthur Andersen LLP (2008), the California Supreme Court held that non-compete "
            "agreements are void under Section 16600, with exceptions only for business sales or partnership "
            "dissolution. The court rejected the narrow restraint doctrine, ruling that any agreement "
            "restraining engagement in a lawful profession is void."
        ),
    },
    {
        "capability": "clause_extraction",
        "input": "Extract the indemnification clause from this contract.",
        "context": (
            "SECTION 4: PAYMENT TERMS. Payment is due within 30 days of invoice. "
            "SECTION 5: INDEMNIFICATION. Vendor shall indemnify, defend, and hold harmless Client and its "
            "officers, directors, and employees from and against any and all claims, damages, losses, and "
            "expenses (including reasonable attorneys' fees) arising out of or relating to (a) Vendor's breach "
            "of this Agreement, (b) Vendor's negligence or willful misconduct, or (c) any infringement of "
            "third-party intellectual property rights by Vendor's deliverables. "
            "SECTION 6: LIMITATION OF LIABILITY. Neither party's aggregate liability shall exceed the total "
            "fees paid in the twelve months preceding the claim."
        ),
        "output": (
            "INDEMNIFICATION (Section 5): Vendor shall indemnify, defend, and hold harmless Client and its "
            "officers, directors, and employees from and against any and all claims, damages, losses, and "
            "expenses (including reasonable attorneys' fees) arising out of or relating to (a) Vendor's breach "
            "of this Agreement, (b) Vendor's negligence or willful misconduct, or (c) any infringement of "
            "third-party intellectual property rights by Vendor's deliverables."
        ),
    },
    {
        "capability": "clause_extraction",
        "input": "Extract the governing law clause.",
        "context": (
            "SECTION 9: FORCE MAJEURE. Neither party shall be liable for delays caused by events beyond "
            "reasonable control. "
            "SECTION 10: GOVERNING LAW AND JURISDICTION. This Agreement shall be governed by and construed "
            "in accordance with the laws of the State of Delaware, without regard to its conflict of laws "
            "principles. Any dispute arising under this Agreement shall be subject to the exclusive jurisdiction "
            "of the state and federal courts located in Wilmington, Delaware. "
            "SECTION 11: ENTIRE AGREEMENT. This Agreement constitutes the entire agreement between the parties."
        ),
        "output": (
            "GOVERNING LAW AND JURISDICTION (Section 10): This Agreement shall be governed by and construed "
            "in accordance with the laws of the State of Delaware, without regard to its conflict of laws "
            "principles. Any dispute arising under this Agreement shall be subject to the exclusive jurisdiction "
            "of the state and federal courts located in Wilmington, Delaware."
        ),
    },
    {
        "capability": "missing_provision_detection",
        "input": "Identify missing standard provisions in this employment agreement.",
        "context": (
            "This Employment Agreement is between Acme Corp and Jane Smith. "
            "COMPENSATION: Base salary of $150,000 per year, paid bi-weekly. "
            "DUTIES: Employee shall serve as Senior Software Engineer and report to the VP of Engineering. "
            "START DATE: Employment begins on March 1, 2025."
        ),
        "output": (
            "The following standard provisions are missing from this employment agreement: "
            "1. Termination clause — no at-will or for-cause termination language. "
            "2. Benefits and PTO — no mention of health insurance, retirement, or vacation policy. "
            "3. Confidentiality/NDA — no protection for proprietary information. "
            "4. Intellectual property assignment — no clause assigning work product to employer. "
            "5. Dispute resolution — no arbitration or litigation venue specified. "
            "6. Non-solicitation — no restriction on soliciting employees or clients post-departure."
        ),
    },
    {
        "capability": "missing_provision_detection",
        "input": "Review this freelance contract for missing provisions.",
        "context": (
            "FREELANCE SERVICES AGREEMENT between Widget Co and John Developer. "
            "SCOPE: Developer will build a customer portal web application. "
            "PAYMENT: $10,000 upon completion. "
            "TIMELINE: Project must be completed within 90 days of signing."
        ),
        "output": (
            "The following standard provisions are missing: "
            "1. Intellectual property ownership — no assignment of IP rights for deliverables. "
            "2. Revision and acceptance criteria — no definition of what constitutes 'completion.' "
            "3. Payment milestones — single lump-sum payment creates risk; no progress payments defined. "
            "4. Termination rights — no clause for early termination by either party. "
            "5. Confidentiality — no NDA protecting either party's proprietary information. "
            "6. Liability limitation — no cap on damages or warranty disclaimers. "
            "7. Independent contractor status — no clarification of employment classification."
        ),
    },
]

print(f"Test dataset: {len(TEST_CASES)} cases across {len(set(c['capability'] for c in TEST_CASES))} capabilities")

Eight test cases covering all four capabilities — two per capability. Each has realistic legal content that an actual legal assistant would encounter.

> **Tip:** For a production pipeline, you'd want 20-50 test cases per capability. Start with 8 to prove the pipeline works, then expand.

## Step 4: Create legal-specific custom evals

The built-in metrics handle general quality — but LexAI has domain-specific requirements that no built-in metric covers. Create two custom evals in the dashboard.

**Custom eval 1: `legal_citation_accuracy`**

1. Go to [app.futureagi.com](https://app.futureagi.com) → **Evals** (left sidebar under BUILD)
2. Click **Create Evaluation**
3. Fill in:
   - **Name**: `legal_citation_accuracy`
   - **Template type**: **Use Future AGI Agents**
   - **Model**: `turing_small`
   - **Output Type**: `Pass/Fail`
4. Write the **Rule Prompt**:

```
You are evaluating a legal document assistant's response for citation accuracy.

The assistant was given this task: {{input}}
The source material is: {{context}}
The assistant responded: {{output}}

Mark PASS only if all of these are true:
- Every case name, statute, or legal reference mentioned in the response appears in the source material
- Case years, court names, and holding descriptions match the source exactly
- No fabricated or hallucinated legal citations are present

Mark FAIL if any legal citation is invented, any case detail is wrong, or any statute is misidentified.

Return a clear PASS/FAIL decision with a reason identifying any citation errors found.
```

5. Click **Create Evaluation**

**Custom eval 2: `contract_completeness`**

Repeat the process with:
- **Name**: `contract_completeness`
- **Output Type**: `Percentage`
- **Rule Prompt**:

```
You are evaluating whether a drafted contract clause includes all standard provisions.

The drafting requirements: {{input}}
The reference material: {{context}}
The drafted clause: {{output}}

Score using these criteria:
- 25 points: All parties and their obligations are clearly identified
- 25 points: All terms from the reference material are accurately incorporated
- 25 points: Standard protective language is included (termination rights, breach remedies, dispute resolution as applicable)
- 25 points: The clause is legally precise — no ambiguous terms, no missing definitions

Return a normalized score from 0.0 to 1.0 (for example, 0.75 for 75/100) with a reason listing any missing elements.
```

Both evals are now registered in the platform and available by name in SDK calls.

> **Note:** See Custom Eval Metrics: Write Your Own Evaluation Criteria for the full custom eval workflow — Pass/Fail vs Percentage output types, Rule Prompt variables, and running custom evals via SDK.

## Step 5: Wire the eval pipeline

Now bring everything together: run all built-in and custom evals across the test dataset in a single Python script. This is your reusable eval pipeline.

In [ ]:
import os
from fi.evals import evaluate, Evaluator

evaluator = Evaluator(
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

# Define which metrics apply to each capability
CAPABILITY_METRICS = {
    "contract_drafting": {
        "builtin": ["completeness", "factual_accuracy", "groundedness"],
        "custom": ["contract_completeness"],
    },
    "case_summarization": {
        "builtin": ["completeness", "context_relevance", "chunk_utilization"],
        "custom": ["legal_citation_accuracy"],
    },
    "clause_extraction": {
        "builtin": ["chunk_attribution", "factual_accuracy"],
        "custom": ["legal_citation_accuracy"],
    },
    "missing_provision_detection": {
        "builtin": ["completeness"],
        "custom": ["contract_completeness"],
    },
}

results = []

for case in TEST_CASES:
    capability = case["capability"]
    metrics = CAPABILITY_METRICS[capability]

    case_results = {
        "capability": capability,
        "input": case["input"][:60] + "...",
        "scores": {},
    }

    # Run built-in metrics
    for metric in metrics["builtin"]:
        result = evaluate(
            metric,
            output=case["output"],
            context=case["context"],
            input=case["input"],
            model="turing_small",
        )
        case_results["scores"][metric] = {
            "score": result.score,
            "passed": result.passed,
            "reason": result.reason,
        }

    # Run custom metrics
    for metric in metrics["custom"]:
        result = evaluator.evaluate(
            eval_templates=metric,
            inputs={
                "input": case["input"],
                "context": case["context"],
                "output": case["output"],
            },
        )
        eval_result = result.eval_results[0]
        score = eval_result.output[0] if isinstance(eval_result.output, list) else eval_result.output
        case_results["scores"][metric] = {
            "score": score,
            "passed": eval_result.output not in ["Fail", "FAIL", False],
            "reason": eval_result.reason,
        }

    results.append(case_results)

# Print summary
print(f"\n{'='*80}")
print(f"  LexAI Eval Pipeline — {len(results)} test cases")
print(f"{'='*80}\n")

for r in results:
    print(f"[{r['capability']}] {r['input']}")
    for metric, data in r["scores"].items():
        status = "PASS" if data["passed"] else "FAIL"
        print(f"  {metric:<30} {status}  (score: {data['score']})")
    print()

The key insight here: built-in metrics use the `evaluate()` function directly, while custom evals use the `Evaluator.evaluate()` method with the eval name you registered in the dashboard. Both return scores and reasons you can inspect.

> **Note:** `evaluate("completeness", ...)` calls the built-in metric directly. `evaluator.evaluate(eval_templates="legal_citation_accuracy", ...)` calls your custom eval via the platform.

## Step 6: Interpret the results

Raw scores aren't useful until you organize them by capability. Build a quality matrix that shows where each capability stands across its relevant metrics.

In [ ]:
from collections import defaultdict

# Aggregate scores by capability and metric
capability_scores = defaultdict(lambda: defaultdict(list))

for r in results:
    for metric, data in r["scores"].items():
        capability_scores[r["capability"]][metric].append(data["passed"])

# Print the quality matrix
print(f"\n{'='*80}")
print(f"  Quality Matrix — Capability x Metric (pass rate)")
print(f"{'='*80}\n")

for capability, metrics in capability_scores.items():
    print(f"  {capability}")
    print(f"  {'-' * 50}")
    for metric, passes in metrics.items():
        rate = sum(passes) / len(passes)
        bar = "#" * int(rate * 20) + "." * (20 - int(rate * 20))
        print(f"    {metric:<30} [{bar}] {rate:.0%}")
    print()

# Identify weakest spots
print(f"\n--- Areas needing attention ---\n")

for capability, metrics in capability_scores.items():
    for metric, passes in metrics.items():
        rate = sum(passes) / len(passes)
        if rate < 1.0:
            print(f"  {capability} / {metric}: {rate:.0%} pass rate")
            # Show the failing cases
            for r in results:
                if r["capability"] == capability and metric in r["scores"]:
                    if not r["scores"][metric]["passed"]:
                        print(f"    Reason: {r['scores'][metric]['reason']}")

This matrix tells you exactly where to focus. If contract drafting scores well on `factual_accuracy` but poorly on `completeness`, you know the agent gets the facts right but misses provisions. If case summarization fails `chunk_utilization`, the agent is only reading the first paragraph of the source material.

Each failure reason gives you a specific fix:
- Low `completeness` on contract drafting → add "include all standard provisions" to the system prompt
- Low `chunk_utilization` on case summarization → add "synthesize from the entire source document, not just the opening paragraph"
- Low `factual_accuracy` on clause extraction → add "reproduce clauses verbatim from the source, do not paraphrase"

> **Note:** For a structured approach to improving prompts based on eval results, see Evaluation-Driven Development — it shows the score-revise-rescore loop with quality gates.

## Step 7: Automate for CI/CD

Once your pipeline is stable, automate it so every prompt change gets evaluated before it ships. The script below is your CI/CD-ready eval runner — save it as `scripts/lexai_eval_pipeline.py`.

In [ ]:
#!/usr/bin/env python3
"""
LexAI eval pipeline for CI/CD.
Exit 0 = all capabilities above threshold. Exit 1 = at least one below.
"""
import os
import sys
from collections import defaultdict
from fi.evals import evaluate, Evaluator

evaluator = Evaluator(
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

# Quality thresholds per metric
THRESHOLDS = {
    "completeness": 0.80,
    "factual_accuracy": 0.90,
    "groundedness": 0.85,
    "context_relevance": 0.80,
    "chunk_utilization": 0.75,
    "chunk_attribution": 0.85,
    "legal_citation_accuracy": 0.90,
    "contract_completeness": 0.75,
}

CAPABILITY_METRICS = {
    "contract_drafting": {
        "builtin": ["completeness", "factual_accuracy", "groundedness"],
        "custom": ["contract_completeness"],
    },
    "case_summarization": {
        "builtin": ["completeness", "context_relevance", "chunk_utilization"],
        "custom": ["legal_citation_accuracy"],
    },
    "clause_extraction": {
        "builtin": ["chunk_attribution", "factual_accuracy"],
        "custom": ["legal_citation_accuracy"],
    },
    "missing_provision_detection": {
        "builtin": ["completeness"],
        "custom": ["contract_completeness"],
    },
}


def run_pipeline(test_cases: list) -> bool:
    capability_scores = defaultdict(lambda: defaultdict(list))

    for case in test_cases:
        capability = case["capability"]
        metrics = CAPABILITY_METRICS[capability]

        for metric in metrics["builtin"]:
            result = evaluate(
                metric,
                output=case["output"],
                context=case["context"],
                input=case["input"],
                model="turing_small",
            )
            capability_scores[capability][metric].append(result.score)

        for metric in metrics["custom"]:
            result = evaluator.evaluate(
                eval_templates=metric,
                inputs={
                    "input": case["input"],
                    "context": case["context"],
                    "output": case["output"],
                },
            )
            eval_result = result.eval_results[0]
            score = eval_result.output[0] if isinstance(eval_result.output, list) else eval_result.output
            capability_scores[capability][metric].append(
                float(score) if score is not None else 0.0
            )

    # Check thresholds
    all_passed = True

    print(f"\n{'Capability':<35} {'Metric':<30} {'Avg Score':>10} {'Threshold':>10} {'Status':>8}")
    print("-" * 97)

    for capability, metrics in capability_scores.items():
        for metric, scores in metrics.items():
            avg = sum(scores) / len(scores) if scores else 0.0
            threshold = THRESHOLDS.get(metric, 0.75)
            passed = avg >= threshold
            status = "PASS" if passed else "FAIL"

            if not passed:
                all_passed = False

            print(f"{capability:<35} {metric:<30} {avg:>10.2f} {threshold:>10.2f} {status:>8}")

    return all_passed


passed = run_pipeline(TEST_CASES)
print(f"\n{'Pipeline PASSED' if passed else 'Pipeline FAILED'}")

**GitHub Actions workflow** — add `.github/workflows/lexai-eval.yml` to your repo:

```yaml
name: LexAI Eval Pipeline

on:
  pull_request:
    branches: [main]
    paths:
      - "prompts/**"
      - "scripts/**"

jobs:
  evaluate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install ai-evaluation futureagi openai
      - name: Run LexAI eval pipeline
        env:
          FI_API_KEY: ${{ secrets.FI_API_KEY }}
          FI_SECRET_KEY: ${{ secrets.FI_SECRET_KEY }}
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
        run: python scripts/lexai_eval_pipeline.py
```

Every PR that touches a prompt file now triggers the full eval pipeline. If any capability drops below its threshold, the merge is blocked.

> **Note:** See CI/CD Eval Pipeline: Automate Quality Gates in GitHub Actions for the full GitHub Actions setup — PR comments, branch protection rules, and secret management.

## What you built

You now have a systematic approach to eval selection and a reusable eval pipeline tailored to your application's specific capabilities — from metric mapping to CI/CD automation.

Here's the methodology, distilled:

```
Describe capabilities → Map to metrics → Build test dataset →
Create custom evals → Wire the pipeline → Interpret results →
Automate in CI/CD
```

The pipeline you built:

- **Mapped 4 capabilities** (contract drafting, case summarization, clause extraction, missing provision detection) to the right built-in metrics
- **Created 2 custom evals** (`legal_citation_accuracy`, `contract_completeness`) for domain-specific quality criteria no built-in metric covers
- **Built a reusable test dataset** with 8 legal-specific test cases covering all capabilities
- **Wired a Python pipeline** that runs built-in + custom evals and produces a quality matrix by capability
- **Automated for CI/CD** with per-metric thresholds and non-zero exit on failure

The same methodology applies to any domain: describe what your app does, identify what "wrong" looks like for each capability, find the metrics that catch those failure modes, and fill gaps with custom evals.